# 08 — Q7 Batch Speech 30s: Debates vs Newscasts

**Question 7:** Can you relate topics discussed in the debates with those identified in the newscasts?

This version intentionally follows the same folder convention as notebook `05_single_newscast_speech_30s_topic_analysis`:

```text
BASE_DIR = Path(".")
FEATURES_DIR = BASE_DIR / "data" / "features"
```

It reads all `*_speech.pkl` files from `data/features`, creates fixed 30-second speech windows for debates and newscasts, assigns topics using the same dictionary, and compares the topic distributions.


## 0. Environment check only

This notebook does **not** install or upgrade packages.

It uses the same loading style as notebook 05:

```python
pd.read_pickle(path)
```

If this fails, the issue is the Python environment used to open the pickles, not the Q7 logic.


In [ ]:
# ------------------------------------------------------------
# Environment check only. No pip install is done here.
# ------------------------------------------------------------
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python executable:", sys.executable)
print("Python version:", sys.version)

for pkg in ["numpy", "pandas", "pyarrow"]:
    try:
        print(f"{pkg}:", version(pkg))
    except PackageNotFoundError:
        print(f"{pkg}: not installed")

print("\nNo package installation is performed in this notebook.")
print("Use the same kernel/environment where notebook 05 can read one speech pickle successfully.")


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import timedelta
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

## Pickle reader with folder fallback

The reader below does not try to repair the environment. It simply tries every available copy of the same `*_speech.pkl` file.

This is safer than changing NumPy/Pandas/PyArrow versions from inside the notebook.

In [ ]:
def read_pickle_compat(path):
    """Read one speech pickle exactly as notebook 05 does."""
    return pd.read_pickle(path)

print("Pickle reader ready: using pandas.read_pickle directly, same as notebook 05.")


## 1. Configuration

This version only uses:

```text
data/features/*_speech.pkl
```

This matches the structure used in notebook 05.

If you run the notebook from another folder, set `USE_ABSOLUTE_FEATURES_PATH = True` and update `ABSOLUTE_FEATURES_DIR`.


In [ ]:
BASE_DIR = Path(".")

# Same relative path used in notebook 05.
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR_RELATIVE = DATA_DIR / "features"

# Optional absolute override. Keep False if the notebook is inside:
# C:\\Users\\henri\\Desktop\\MEEC_P4\\Biggie Data\\Big Data
USE_ABSOLUTE_FEATURES_PATH = False
ABSOLUTE_FEATURES_DIR = Path(r"C:\Users\henri\Desktop\MEEC_P4\Biggie Data\Big Data\data\features")

FEATURES_DIR = ABSOLUTE_FEATURES_DIR if USE_ABSOLUTE_FEATURES_PATH else FEATURES_DIR_RELATIVE

OUTPUT_DIR = BASE_DIR / "outputs_part3_q7_debate_newscast_topic_relation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main methodological choice for Q7.
SPEECH_WINDOW_SECONDS = 30

# The main comparison excludes Other/Unknown, but the quality report keeps it.
EXCLUDE_UNKNOWN_FROM_MAIN_COMPARISON = True

# Temporal window around debate dates.
TEMPORAL_WINDOW_DAYS = 3

# Use all files. Set to a small integer only for debugging.
MAX_FILES = None

# If True, files that are neither detected as debate nor newscast are ignored.
IGNORE_UNKNOWN_VIDEO_TYPES = True

print("Current working directory:", Path.cwd())
print("Features folder:", FEATURES_DIR.resolve())
print("Features folder exists:", FEATURES_DIR.exists())
print("Output folder:", OUTPUT_DIR.resolve())

if not FEATURES_DIR.exists():
    raise FileNotFoundError(
        f"FEATURES_DIR does not exist: {FEATURES_DIR.resolve()}\n"
        "Run this notebook from the project root folder, or set USE_ABSOLUTE_FEATURES_PATH=True."
    )


## 2. Text cleaning and topic dictionary

This uses the same type of dictionary-based topic detection as the single-newscast speech notebook.

The important point for Q7 is that **the exact same dictionary is applied to debates and newscasts**.

In [ ]:
STOPWORDS_PT = {
    "de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "com", "não", "nao", "uma", "os", "no", "se", "na",
    "por", "mais", "as", "dos", "como", "mas", "foi", "ao", "ele", "das", "tem", "à", "aos", "seu", "sua",
    "ou", "ser", "quando", "muito", "há", "ha", "nos", "já", "ja", "está", "esta", "estao", "estão",
    "entre", "também", "tambem", "só", "so", "pelo", "pela", "até", "ate", "isso", "este", "esta",
    "num", "numa", "mesmo", "assim", "sobre", "ainda", "foram", "será", "sera", "ter", "têm", "tem",
    "vai", "vão", "vao", "pode", "podem", "porque", "onde", "depois", "todos", "todas"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "noticias", "notícias", "rtp", "sic", "tvi", "cnn", "cm", "cmtv",
    "direto", "directo", "minuto", "minutos", "hora", "horas", "hoje", "amanha", "amanhã",
    "ontem", "agora", "imagem", "imagens", "fonte", "arquivo"
}


def clean_text_pt(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    text = clean_text_pt(text)
    tokens = text.split()
    tokens = [t for t in tokens if len(t) >= min_len]
    tokens = [t for t in tokens if not t.isdigit()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens


def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return None
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

In [ ]:
# ------------------------------------------------------------
# Candidate / party / theme aliases
# ------------------------------------------------------------

CANDIDATE_ALIASES = {
    "André Ventura": ["andre ventura", "ventura", "lider do chega"],
    "Cotrim Figueiredo": ["cotrim", "cotrim figueiredo", "cotrim de figueiredo", "joao cotrim de figueiredo"],
    "Luís Marques Mendes": ["marques mendes", "luis marques mendes"],
    "Henrique Gouveia e Melo": ["gouveia e melo", "gouveia melo", "henrique gouveia e melo", "almirante gouveia e melo"],
    "António José Seguro": ["antonio jose seguro", "jose seguro", "antonio seguro"],
    "António Filipe": ["antonio filipe"],
    "Catarina Martins": ["catarina martins"],
    "Jorge Pinto": ["jorge pinto"],
}

PARTY_ALIASES = {
    "CHEGA": ["chega", "partido chega"],
    "IL": ["il", "iniciativa liberal", "liberais"],
    "PSD/AD": ["psd", "ad", "alianca democratica", "partido social democrata", "sociais democratas"],
    "PS": ["ps", "partido socialista", "socialistas"],
    "PCP/CDU": ["pcp", "cdu", "partido comunista", "comunistas"],
    "BE": ["be", "bloco de esquerda"],
    "LIVRE": ["livre", "partido livre"],
    "CDS": ["cds", "cds pp", "cds-pp", "centro democratico social"],
}

THEME_ALIASES = {
    "Abertura/Manchetes": [
        "manchetes", "destaques", "sumário", "sumario",
        "em destaque", "abertura", "boa noite"
    ],

    "Eleições/Campanha": [
        "presidenciais", "presidencial",
        "eleições", "eleicoes", "eleição", "eleicao",
        "eleições presidenciais", "eleicoes presidenciais",
        "candidato", "candidata", "candidatos", "candidaturas",
        "campanha", "campanha eleitoral",
        "debate eleitoral", "debates eleitorais",
        "voto", "votos", "eleitores", "urna", "urnas"
    ],

    "Sondagens": [
        "sondagem", "sondagens",
        "barómetro", "barometro",
        "intenção de voto", "intencao de voto",
        "intenções de voto", "intencoes de voto",
        "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral",
        "pontos percentuais"
    ],

    "Governo/Partidos": [
        "governo", "executivo",
        "primeiro ministro", "primeiro-ministro",
        "ministro", "ministra", "ministros",
        "parlamento", "assembleia da república", "assembleia da republica",
        "oposição", "oposicao",
        "líder parlamentar", "lider parlamentar",
        "grupo parlamentar", "maioria absoluta",
        "moção de censura", "mocao de censura",
        "partidos políticos", "partidos politicos"
    ],

    "Saúde": [
        "saúde", "saude",
        "sns", "serviço nacional de saúde", "servico nacional de saude",
        "hospital", "hospitais",
        "médico", "medico", "médicos", "medicos",
        "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras",
        "urgência", "urgencia", "urgências", "urgencias",
        "doente", "doentes", "utente", "utentes",
        "vacina", "vacinas", "covid",
        "lista de espera", "listas de espera"
    ],

    "Economia/Proteção Social": [
        "economia", "económico", "economico",
        "inflação", "inflacao",
        "preços", "precos", "custo de vida",
        "impostos", "irs", "iva", "irc",
        "orçamento", "orcamento", "orçamento do estado", "orcamento do estado",
        "défice", "defice", "dívida pública", "divida publica",
        "salário", "salario", "salários", "salarios",
        "salário mínimo", "salario minimo",
        "pensões", "pensoes", "reformas",
        "segurança social", "seguranca social",
        "subsídio", "subsidio", "subsídios", "subsidios",
        "apoios sociais", "apoio social",
        "empresas", "juros", "taxas de juro",
        "banco de portugal", "bce", "bancos"
    ],

    "Habitação": [
        "habitação", "habitacao",
        "arrendamento", "arrendar",
        "renda", "rendas",
        "senhorio", "senhorios",
        "inquilino", "inquilinos",
        "crédito habitação", "credito habitacao",
        "empréstimo da casa", "emprestimo da casa",
        "preço das casas", "precos das casas",
        "mercado imobiliário", "mercado imobiliario",
        "imobiliário", "imobiliario"
    ],

    "Educação": [
        "educação", "educacao",
        "escola", "escolas",
        "professor", "professores",
        "aluno", "alunos",
        "ensino", "aulas",
        "creche", "creches",
        "universidade", "universidades",
        "estudante", "estudantes",
        "exames nacionais", "ano letivo", "ano lectivo"
    ],

    "Justiça/Segurança": [
        "justiça", "justica",
        "tribunal", "tribunais",
        "polícia", "policia",
        "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria",
        "crime", "crimes", 
        "fraude", "fraudes",
        "sócrates", "socrates",
        "josé sócrates", "jose socrates",
        "homicídio", "homicidio",
        "agressão", "agressao",
        "corrupção", "corrupcao",
        "pgr", "ministério público", "ministerio publico",
        "detido", "detidos", "arguido", "arguidos",
        "prisão", "prisao",
        "caução", "caucao",
        "buscas", "operação policial", "operacao policial",

        # Acidentes e segurança pública
        "acidente", "acidentes",
        "colisão", "colisao",
        "despiste",
        "queda",
        "descarrilamento",
        "vítima", "vitima", "vítimas", "vitimas",
        "ferido", "feridos", "ferida", "feridas",
        "morto", "mortos", "morta", "mortas",
        "falha de segurança", "falha de seguranca",
        "falha técnica", "falha tecnica",
        "investigação", "investigacao",
        "inquérito", "inquerito"
    ],

    "Internacional": [
        "ucrânia", "ucrania",
        "rússia", "russia",
        "guerra na ucrânia", "guerra na ucrania",
        "israel", "gaza", "palestina", "hamas",
        "médio oriente", "medio oriente",
        "trump", "donald trump",
        "casa branca",
        "eua", "estados unidos",
        "bruxelas", "união europeia", "uniao europeia",
        "diplomacia", "diplomacia europeia",
        "diplomático", "diplomatico",
        "diplomática", "diplomatica",
        "relações diplomáticas", "relacoes diplomaticas",
        "nato", "onu",
        "frança", "franca",
        "espanha", "brasil", "china",
        "reino unido", "alemanha",
        "áfrica do sul", "africa do sul",
        "g20", "g7",

        # Política norte-americana
        "republicanos", "republicano",
        "partido republicano",
        "democratas", "democrata",
        "partido democrata",
        "congresso americano",
        "senado americano",
        "câmara dos representantes", "camara dos representantes"
    ],

    "Greves/Trabalho": [
        "greve", "greves",
        "sindicato", "sindicatos",
        "trabalhador", "trabalhadores",
        "protesto", "protestos",
        "manifestação", "manifestacao",
        "manifestantes",
        "contrato coletivo", "contrato colectivo",
        "concertação social", "concertacao social"
    ],

    "Transportes/Mobilidade": [
        "transportes",
        "metro", "metropolitano",
        "comboio", "comboios",
        "cp", "fertagus",
        "autocarro", "autocarros",
        "trânsito", "transito",
        "aeroporto",
        "tap", "tap air portugal",
        "avião", "aviao", "aviões", "avioes",
        "estrada", "autoestrada",
        "portagens",

        # Operadores e transportes urbanos
        "carris",
        "elétrico", "eletrico",
        "ascensor", "ascensores",
        "funicular",

        # Calçada/Elevador da Glória
        "calçada da glória", "calcada da gloria",
        "elevador da glória", "elevador da gloria"
    ],

    "Ambiente/Meteorologia/Proteção Civil": [
        "ambiente",
        "clima", "climático", "climatico",
        "alterações climáticas", "alteracoes climaticas",
        "seca", "chuva", "temporal",
        "inundações", "inundacoes", "cheias",
        "emissões", "emissoes", "poluição", "poluicao",
        "meteorologia", "previsão meteorológica", "previsao meteorologica",
        "temperatura", "vento", "frio", "calor", "neve",
        "incêndio", "incendio", "incêndios", "incendios",
        "bombeiros",
        "proteção civil", "protecao civil",
        "chamas", "evacuação", "evacuacao"
    ],

    "Desporto": [
        "futebol",
        "benfica", "sporting", "fc porto",
        "liga dos campeões", "liga dos campeoes",
        "liga portuguesa",
        "campeonato nacional",
        "seleção nacional", "selecao nacional",
        "treinador", "jogador", "jogadores",
        "golo", "golos",
        "estádio", "estadio"
    ],

    "Cultura": [
        "cultura",
        "cinema", "teatro",
        "música", "musica",
        "festival", "festivais",
        "livro", "livros",
        "exposição", "exposicao",
        "museu", "museus",
        "artista", "artistas",
        "concerto", "concertos"
    ],
}


def normalize_alias(alias):
    return clean_text_pt(alias)


def contains_alias(text, aliases):
    if not isinstance(text, str):
        return False
    clean = clean_text_pt(text)
    for alias in aliases:
        alias_norm = normalize_alias(alias)
        if not alias_norm:
            continue
        pattern = r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)"
        if re.search(pattern, clean):
            return True
    return False


def count_alias_hits(text, alias_dict):
    clean = clean_text_pt(text)
    counts = {}
    for label, aliases in alias_dict.items():
        total = 0
        for alias in aliases:
            alias_norm = normalize_alias(alias)
            if not alias_norm:
                continue
            pattern = r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)"
            total += len(re.findall(pattern, clean))
        counts[label] = int(total)
    return counts


def labels_present(text, alias_dict):
    counts = count_alias_hits(text, alias_dict)
    return [label for label, value in counts.items() if value > 0]


def concat_unique_non_empty(values, max_items=10):
    out = []
    for value in values:
        if pd.isna(value) or str(value).strip() == "":
            continue
        for item in str(value).split(","):
            item = item.strip()
            if item and item not in out:
                out.append(item)
            if len(out) >= max_items:
                break
        if len(out) >= max_items:
            break
    return ", ".join(out)


def split_items(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {x.strip() for x in str(value).split(",") if x.strip()}


def theme_group(theme):
    politics = {"Eleições/Campanha", "Sondagens", "Governo/Partidos"}
    social = {"Saúde", "Educação", "Habitação", "Economia/Proteção Social"}
    risk = {"Justiça/Segurança", "Ambiente/Meteorologia/Proteção Civil"}

    if theme in politics:
        return "politics"
    if theme in social:
        return "social"
    if theme in risk:
        return "risk"
    return theme

print("N candidates:", len(CANDIDATE_ALIASES))
print("N parties:", len(PARTY_ALIASES))
print("N themes:", len(THEME_ALIASES))

## 3. Metadata helpers

These functions infer:

- whether a file is a debate or a newscast;
- the TV channel when available;
- the date from filenames such as `Nov_20`, `November_17`, `Dec_2`, etc.

In [ ]:
MONTH_MAP = {
    "jan": 1, "january": 1, "janeiro": 1,
    "feb": 2, "february": 2, "fevereiro": 2,
    "mar": 3, "march": 3, "marco": 3, "março": 3,
    "apr": 4, "april": 4, "abril": 4,
    "may": 5, "maio": 5,
    "jun": 6, "june": 6, "junho": 6,
    "jul": 7, "july": 7, "julho": 7,
    "aug": 8, "august": 8, "agosto": 8,
    "sep": 9, "sept": 9, "september": 9, "setembro": 9,
    "oct": 10, "october": 10, "outubro": 10,
    "nov": 11, "november": 11, "novembro": 11,
    "dec": 12, "december": 12, "dez": 12, "dezembro": 12,
}

MONTH_LABEL = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}


def classify_source_type(video_id):
    name = str(video_id).lower()
    if "telejornal" in name or "newscast" in name:
        return "newscast"
    if "_vs_" in name or " vs " in name or "debate" in name:
        return "debate"
    return "unknown"


def extract_channel(video_id):
    name = str(video_id).upper()
    for ch in ["RTP", "TVI", "SIC", "CNN", "CMTV", "CM"]:
        if re.search(rf"(^|_){ch}($|_)", name) or ch in name:
            return ch
    return "Unknown"


def extract_date_from_video_id(video_id):
    """Return pandas Timestamp or NaT from names like Nov_20, November_17, Dec_2."""
    name = clean_text_pt(str(video_id)).replace(" ", "_")
    pattern = r"(jan(?:uary)?|janeiro|feb(?:ruary)?|fevereiro|mar(?:ch)?|marco|março|apr(?:il)?|abril|may|maio|jun(?:e)?|junho|jul(?:y)?|julho|aug(?:ust)?|agosto|sep(?:t)?(?:ember)?|setembro|oct(?:ober)?|outubro|nov(?:ember)?|novembro|dec(?:ember)?|dez(?:embro)?)[_\- ]+(\d{1,2})"
    match = re.search(pattern, name)
    if not match:
        return pd.NaT

    month_token = match.group(1)
    day = int(match.group(2))

    # Normalize month abbreviations.
    month = None
    for key, value in MONTH_MAP.items():
        if month_token.startswith(key):
            month = value
            break

    if month is None:
        return pd.NaT

    # Dataset is around Nov/Dec 2025 and Jan 2026.
    year = 2026 if month in {1, 2} else 2025

    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except Exception:
        return pd.NaT


def date_key_from_timestamp(ts):
    if pd.isna(ts):
        return "UnknownDate"
    return f"{MONTH_LABEL[int(ts.month)]}_{int(ts.day)}"


def readable_video_label(video_id):
    label = str(video_id)
    label = label.replace("_speech", "")
    label = re.sub(r"_(November|Nov|December|Dec|January|Jan)_\d+", "", label, flags=re.IGNORECASE)
    label = label.replace("_", " ")
    return label


def is_unknown_theme(theme):
    return str(theme).strip().lower() in {"", "other", "unknown", "other/unknown", "no speech", "nan", "none"}

## 4. Speech-processing helpers

This is the generic version of the 30-second speech-window logic.

It does not perform newscast-specific transition detection. It only creates comparable 30-second topic windows for every speech file.

In [ ]:
def infer_speech_columns(df):
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    start_candidates = ["timestamp", "start", "start_time", "start_sec", "start_seconds", "time"]
    duration_candidates = ["duration", "dur", "duration_sec", "duration_seconds"]
    end_candidates = ["end", "end_time", "end_sec", "end_seconds"]
    transcript_candidates = ["transcript", "text", "whisper", "sentence", "content"]

    def pick(candidates):
        for c in candidates:
            if c in lower:
                return lower[c]
        return None

    start_col = pick(start_candidates)
    duration_col = pick(duration_candidates)
    end_col = pick(end_candidates)
    transcript_col = pick(transcript_candidates)

    if start_col is None:
        raise ValueError(f"Could not infer speech start column. Available columns: {cols}")
    if transcript_col is None:
        raise ValueError(f"Could not infer transcript column. Available columns: {cols}")

    return start_col, duration_col, end_col, transcript_col


def normalise_speech_dataframe(raw_df):
    df = raw_df.copy()
    start_col, duration_col, end_col, transcript_col = infer_speech_columns(df)

    out = pd.DataFrame()
    out["segment_id"] = np.arange(1, len(df) + 1)
    out["start_sec"] = pd.to_numeric(df[start_col], errors="coerce")
    out["transcript"] = df[transcript_col].fillna("").astype(str)

    if duration_col is not None:
        out["duration_sec"] = pd.to_numeric(df[duration_col], errors="coerce")
        out["end_sec"] = out["start_sec"] + out["duration_sec"]
    elif end_col is not None:
        out["end_sec"] = pd.to_numeric(df[end_col], errors="coerce")
        out["duration_sec"] = out["end_sec"] - out["start_sec"]
    else:
        out = out.sort_values("start_sec").reset_index(drop=True)
        out["end_sec"] = out["start_sec"].shift(-1)
        out.loc[out.index[-1], "end_sec"] = out.loc[out.index[-1], "start_sec"] + 5
        out["duration_sec"] = out["end_sec"] - out["start_sec"]

    out = out.dropna(subset=["start_sec", "end_sec", "duration_sec"]).copy()
    out = out[out["transcript"].str.strip() != ""].copy()
    out = out.sort_values("start_sec").reset_index(drop=True)

    # Fix impossible durations conservatively.
    invalid = out["duration_sec"] <= 0
    if invalid.any():
        out.loc[invalid, "duration_sec"] = 1.0
        out.loc[invalid, "end_sec"] = out.loc[invalid, "start_sec"] + 1.0

    out["start_sec"] = out["start_sec"].astype(float)
    out["end_sec"] = out["end_sec"].astype(float)
    out["duration_sec"] = out["duration_sec"].astype(float)
    out["n_words"] = out["transcript"].apply(lambda x: len(str(x).split()))

    return out


def split_transcript_words(text):
    if pd.isna(text):
        return []
    return re.findall(r"\S+", str(text))


def build_30s_windows_for_video(speech_df, video_id, source_path):
    speech = normalise_speech_dataframe(speech_df)

    source_type = classify_source_type(video_id)
    channel = extract_channel(video_id)
    date = extract_date_from_video_id(video_id)
    date_key = date_key_from_timestamp(date)

    if speech.empty:
        return pd.DataFrame(), pd.DataFrame(), {
            "video_id": video_id,
            "source_path": str(source_path),
            "source_type": source_type,
            "channel": channel,
            "date": date,
            "date_key": date_key,
            "status": "empty_after_normalisation",
        }

    word_rows = []

    for _, row in speech.iterrows():
        words = split_transcript_words(row["transcript"])
        if len(words) == 0:
            continue

        start = float(row["start_sec"])
        duration = max(float(row["duration_sec"]), 1.0)

        for i, word in enumerate(words):
            approx_sec = start + ((i + 0.5) / len(words)) * duration
            ws = int(np.floor(approx_sec / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)
            we = ws + SPEECH_WINDOW_SECONDS
            word_rows.append({
                "segment_id": int(row["segment_id"]),
                "word": word,
                "approx_sec": approx_sec,
                "window_start_sec": ws,
                "window_end_sec": we,
            })

    speech_words = pd.DataFrame(word_rows)
    if speech_words.empty:
        return pd.DataFrame(), pd.DataFrame(), {
            "video_id": video_id,
            "source_path": str(source_path),
            "source_type": source_type,
            "channel": channel,
            "date": date,
            "date_key": date_key,
            "status": "no_words",
        }

    timeline_start = int(np.floor(float(speech["start_sec"].min()) / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)
    timeline_end = int(np.ceil(float(speech["end_sec"].max()) / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)
    window_starts = list(range(timeline_start, timeline_end, SPEECH_WINDOW_SECONDS))

    grouped_words = dict(tuple(speech_words.groupby("window_start_sec")))
    window_rows = []
    theme_long_rows = []

    for local_window_id, ws in enumerate(window_starts, start=1):
        we = ws + SPEECH_WINDOW_SECONDS
        group = grouped_words.get(ws, pd.DataFrame(columns=speech_words.columns))

        # Keep only windows with actual speech words for a spoken-time comparison.
        if len(group) == 0:
            continue

        text = " ".join(group["word"].astype(str).tolist())
        clean = clean_text_pt(text)
        tokens = tokenize_pt(clean)

        theme_counts = count_alias_hits(clean, THEME_ALIASES)
        candidate_counts = count_alias_hits(clean, CANDIDATE_ALIASES)
        party_counts = count_alias_hits(clean, PARTY_ALIASES)

        theme_hits_total = sum(theme_counts.values())
        if theme_hits_total > 0:
            dominant_theme = max(theme_counts, key=theme_counts.get)
            dominant_theme_score = int(theme_counts[dominant_theme])
        else:
            dominant_theme = "Other/Unknown"
            dominant_theme_score = 0

        themes_present = ", ".join([k for k, v in theme_counts.items() if v > 0])
        candidates_present = ", ".join([k for k, v in candidate_counts.items() if v > 0])
        parties_present = ", ".join([k for k, v in party_counts.items() if v > 0])
        source_segments = sorted(group["segment_id"].dropna().astype(int).unique().tolist())

        window_rows.append({
            "video_id": video_id,
            "video_label": readable_video_label(video_id),
            "source_path": str(source_path),
            "source_type": source_type,
            "source_family": "Debates" if source_type == "debate" else ("Newscasts" if source_type == "newscast" else "Unknown"),
            "channel": channel,
            "date": date,
            "date_key": date_key,
            "window_id": local_window_id,
            "global_window_uid": f"{video_id}__w{local_window_id}",
            "start_sec": ws,
            "end_sec": we,
            "start_time": seconds_to_hhmmss(ws),
            "end_time": seconds_to_hhmmss(we),
            "duration_sec": SPEECH_WINDOW_SECONDS,
            "duration_min": SPEECH_WINDOW_SECONDS / 60,
            "n_raw_words": int(len(group)),
            "n_tokens": int(len(tokens)),
            "source_segments": ", ".join(map(str, source_segments)),
            "dominant_theme": dominant_theme,
            "dominant_theme_score": dominant_theme_score,
            "theme_hits_total": int(theme_hits_total),
            "themes_present": themes_present,
            "candidates_present": candidates_present,
            "parties_present": parties_present,
            "text": text,
            "clean_text": clean,
        })

        for theme, hits in theme_counts.items():
            if hits > 0:
                theme_long_rows.append({
                    "video_id": video_id,
                    "source_type": source_type,
                    "source_family": "Debates" if source_type == "debate" else ("Newscasts" if source_type == "newscast" else "Unknown"),
                    "channel": channel,
                    "date": date,
                    "date_key": date_key,
                    "window_id": local_window_id,
                    "start_sec": ws,
                    "end_sec": we,
                    "theme": theme,
                    "hits": int(hits),
                })

    windows = pd.DataFrame(window_rows)
    theme_long = pd.DataFrame(theme_long_rows)

    summary = {
        "video_id": video_id,
        "video_label": readable_video_label(video_id),
        "source_path": str(source_path),
        "source_type": source_type,
        "source_family": "Debates" if source_type == "debate" else ("Newscasts" if source_type == "newscast" else "Unknown"),
        "channel": channel,
        "date": date,
        "date_key": date_key,
        "status": "ok",
        "n_segments": len(speech),
        "n_windows_with_speech": len(windows),
        "speech_duration_min_from_windows": len(windows) * SPEECH_WINDOW_SECONDS / 60,
        "raw_start_sec": float(speech["start_sec"].min()),
        "raw_end_sec": float(speech["end_sec"].max()),
        "raw_duration_min": float((speech["end_sec"].max() - speech["start_sec"].min()) / 60),
        "n_theme_hits": int(windows["theme_hits_total"].sum()) if not windows.empty else 0,
        "unknown_windows": int((windows["dominant_theme"] == "Other/Unknown").sum()) if not windows.empty else 0,
    }

    return windows, theme_long, summary

## 5. Discover all speech files

The notebook searches automatically and removes duplicate filenames.

In [ ]:
# ------------------------------------------------------------
# Locate speech files in the same folder used by notebook 05.
# ------------------------------------------------------------
speech_files = sorted(FEATURES_DIR.glob("*_speech.pkl"))

speech_items = []
for path in speech_files:
    video_id = path.stem.replace("_speech", "")
    source_type = classify_source_type(video_id)

    if IGNORE_UNKNOWN_VIDEO_TYPES and source_type == "unknown":
        continue

    speech_items.append({
        "filename": path.name,
        "video_id": video_id,
        "path": path,
        "source_type": source_type,
    })

if MAX_FILES is not None:
    speech_items = speech_items[:MAX_FILES]

print("Speech files found in FEATURES_DIR:", len(speech_files))
print("Speech items to process:", len(speech_items))

for item in speech_items[:80]:
    video_id = item["video_id"]
    print(
        f"- {item['filename']:55s} | type={classify_source_type(video_id):9s} "
        f"| channel={extract_channel(video_id):7s} "
        f"| date={date_key_from_timestamp(extract_date_from_video_id(video_id))} "
        f"| path={item['path']}"
    )

if len(speech_items) == 0:
    raise FileNotFoundError(
        f"No usable *_speech.pkl files found in {FEATURES_DIR.resolve()}."
    )


## 6. Process all speech files into 30-second topic windows

This is the batch equivalent of the single-video speech logic.

Every debate and every newscast is processed in the same way.

In [ ]:
all_window_dfs = []
all_theme_long_dfs = []
video_summaries = []
errors = []

for idx, item in enumerate(speech_items, start=1):
    filename = item["filename"]
    video_id = item["video_id"]
    source_type = item["source_type"]
    path = item["path"]

    print(f"[{idx}/{len(speech_items)}] Processing {filename} ({source_type})")

    try:
        # Same direct reading method used in notebook 05.
        raw = read_pickle_compat(path)

        if not isinstance(raw, pd.DataFrame):
            raise TypeError(f"Pickle is not a DataFrame: {type(raw)}")

        print(f"   OK read from: {path}")

        windows, theme_long, summary = build_30s_windows_for_video(raw, video_id, path)
        video_summaries.append(summary)

        if not windows.empty:
            all_window_dfs.append(windows)
        if not theme_long.empty:
            all_theme_long_dfs.append(theme_long)

    except Exception as e:
        errors.append({
            "video_id": video_id,
            "filename": filename,
            "source_path": str(path),
            "error": repr(e),
        })
        print("   ERROR:", repr(e))

all_windows = pd.concat(all_window_dfs, ignore_index=True) if all_window_dfs else pd.DataFrame()
all_theme_long = pd.concat(all_theme_long_dfs, ignore_index=True) if all_theme_long_dfs else pd.DataFrame()
video_summary = pd.DataFrame(video_summaries)
processing_errors = pd.DataFrame(errors)

print("\nDone.")
print("All windows:", all_windows.shape)
print("All theme long:", all_theme_long.shape)
print("Video summary:", video_summary.shape)
print("Errors:", processing_errors.shape)

processing_errors.to_csv(OUTPUT_DIR / "q7_processing_errors.csv", index=False)

if not processing_errors.empty:
    display(Markdown("### Processing errors"))
    display(processing_errors)

if all_windows.empty:
    raise ValueError(
        "No windows were created. This means all speech pickles failed before/inside processing. "
        "First test whether notebook 05 can still read one speech pickle in this same kernel/environment."
    )

with pd.option_context("display.max_rows", 30, "display.max_columns", None, "display.max_colwidth", 220):
    display(all_windows.head(30))
    display(video_summary)


## 7. Export base Q7 dataset

These are the most important intermediate outputs.

They allow the rest of the analysis to be reproduced without re-reading the pickles.

In [ ]:
all_windows.to_csv(OUTPUT_DIR / "q7_all_speech_30s_windows.csv", index=False)
all_theme_long.to_csv(OUTPUT_DIR / "q7_all_speech_30s_theme_long.csv", index=False)
video_summary.to_csv(OUTPUT_DIR / "q7_video_processing_summary.csv", index=False)
processing_errors.to_csv(OUTPUT_DIR / "q7_processing_errors.csv", index=False)

print("Saved base outputs:")
print("-", OUTPUT_DIR / "q7_all_speech_30s_windows.csv")
print("-", OUTPUT_DIR / "q7_all_speech_30s_theme_long.csv")
print("-", OUTPUT_DIR / "q7_video_processing_summary.csv")

## 8. Quality check: Unknown / Other windows

This does not answer the research question directly, but it tells us how much speech could not be confidently mapped to one of the predefined themes.

In [ ]:
quality_by_source = (
    all_windows
    .assign(is_unknown_theme=all_windows["dominant_theme"].apply(is_unknown_theme))
    .groupby(["source_family", "is_unknown_theme"])
    .agg(
        n_windows=("global_window_uid", "count"),
        duration_min=("duration_min", "sum"),
    )
    .reset_index()
)

source_totals = (
    quality_by_source
    .groupby("source_family")["duration_min"]
    .sum()
    .rename("source_total_min")
    .reset_index()
)

quality_by_source = quality_by_source.merge(source_totals, on="source_family", how="left")
quality_by_source["share_pct"] = (100 * quality_by_source["duration_min"] / quality_by_source["source_total_min"].replace(0, np.nan)).round(2)

quality_by_video = (
    all_windows
    .assign(is_unknown_theme=all_windows["dominant_theme"].apply(is_unknown_theme))
    .groupby(["video_id", "source_family", "channel", "date_key", "is_unknown_theme"])
    .agg(n_windows=("global_window_uid", "count"), duration_min=("duration_min", "sum"))
    .reset_index()
)

video_totals = quality_by_video.groupby("video_id")["duration_min"].sum().rename("video_total_min").reset_index()
quality_by_video = quality_by_video.merge(video_totals, on="video_id", how="left")
quality_by_video["share_pct"] = (100 * quality_by_video["duration_min"] / quality_by_video["video_total_min"].replace(0, np.nan)).round(2)

quality_by_source.to_csv(OUTPUT_DIR / "q7_unknown_theme_quality_by_source.csv", index=False)
quality_by_video.to_csv(OUTPUT_DIR / "q7_unknown_theme_quality_by_video.csv", index=False)

display(quality_by_source)

## 9. Global comparison: Debates vs Newscasts

This is the first direct answer to Q7.

It compares the percentage of speech time dedicated to each theme in debates and in newscasts.

In [ ]:
comparison_windows = all_windows.copy()

if EXCLUDE_UNKNOWN_FROM_MAIN_COMPARISON:
    comparison_windows = comparison_windows[~comparison_windows["dominant_theme"].apply(is_unknown_theme)].copy()

# Keep only debates and newscasts for the main comparison.
comparison_windows = comparison_windows[comparison_windows["source_family"].isin(["Debates", "Newscasts"])].copy()

if comparison_windows.empty:
    raise ValueError("No known-theme debate/newscast windows available for comparison.")

global_theme_share = (
    comparison_windows
    .groupby(["source_family", "dominant_theme"])
    .agg(
        n_windows=("global_window_uid", "count"),
        duration_min=("duration_min", "sum"),
        theme_hits=("theme_hits_total", "sum"),
    )
    .reset_index()
)

global_totals = (
    global_theme_share
    .groupby("source_family")["duration_min"]
    .sum()
    .rename("source_total_known_theme_min")
    .reset_index()
)

global_theme_share = global_theme_share.merge(global_totals, on="source_family", how="left")
global_theme_share["share_pct"] = (
    100 * global_theme_share["duration_min"] /
    global_theme_share["source_total_known_theme_min"].replace(0, np.nan)
).round(2)

comparison_pivot = (
    global_theme_share
    .pivot_table(index="dominant_theme", columns="source_family", values="share_pct", fill_value=0)
    .reset_index()
)

for col in ["Debates", "Newscasts"]:
    if col not in comparison_pivot.columns:
        comparison_pivot[col] = 0.0

comparison_pivot["abs_diff_pct_points"] = (comparison_pivot["Debates"] - comparison_pivot["Newscasts"]).abs().round(2)
comparison_pivot["relation"] = np.where(
    (comparison_pivot["Debates"] > 0) & (comparison_pivot["Newscasts"] > 0),
    "present_in_both",
    np.where(comparison_pivot["Debates"] > 0, "debates_only", "newscasts_only")
)
comparison_pivot = comparison_pivot.sort_values(["Debates", "Newscasts"], ascending=False).reset_index(drop=True)

global_theme_share.to_csv(OUTPUT_DIR / "q7_global_theme_share_long.csv", index=False)
comparison_pivot.to_csv(OUTPUT_DIR / "q7_debates_vs_newscasts_theme_share_pivot.csv", index=False)

display(Markdown("### Global theme share — long table"))
display(global_theme_share.sort_values(["source_family", "share_pct"], ascending=[True, False]))

display(Markdown("### Debates vs Newscasts — pivot table"))
display(comparison_pivot)

## 10. Overlap metrics

These metrics provide a compact quantitative answer:

- **Jaccard overlap**: how many themes appear in both sets.
- **Cosine similarity**: how similar the theme-share distributions are.

In [ ]:
def cosine_for_two_vectors(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return np.nan
    return float(np.dot(a, b) / denom)


debate_themes = set(comparison_pivot.loc[comparison_pivot["Debates"] > 0, "dominant_theme"])
newscast_themes = set(comparison_pivot.loc[comparison_pivot["Newscasts"] > 0, "dominant_theme"])
common_themes = debate_themes & newscast_themes
union_themes = debate_themes | newscast_themes

jaccard_overlap = len(common_themes) / max(len(union_themes), 1)
cosine_similarity_themes = cosine_for_two_vectors(comparison_pivot["Debates"], comparison_pivot["Newscasts"])

metrics = pd.DataFrame([{
    "n_debate_themes": len(debate_themes),
    "n_newscast_themes": len(newscast_themes),
    "n_common_themes": len(common_themes),
    "jaccard_overlap": round(jaccard_overlap, 4),
    "cosine_similarity_theme_shares": round(cosine_similarity_themes, 4) if not pd.isna(cosine_similarity_themes) else np.nan,
    "common_themes": ", ".join(sorted(common_themes)),
}])

metrics.to_csv(OUTPUT_DIR / "q7_debate_newscast_overlap_metrics.csv", index=False)
display(metrics)

## 11. Plot: Debates vs Newscasts theme shares

This is the main visual answer for Q7.

In [ ]:
plot_df = comparison_pivot.copy()
plot_df["total_share"] = plot_df["Debates"] + plot_df["Newscasts"]
plot_df = plot_df.sort_values("total_share", ascending=False).drop(columns="total_share")

x = np.arange(len(plot_df))
width = 0.38

plt.figure(figsize=(max(10, len(plot_df) * 0.65), 5))
plt.bar(x - width / 2, plot_df["Debates"], width, label="Debates")
plt.bar(x + width / 2, plot_df["Newscasts"], width, label="Newscasts")
plt.title("Q7 — Theme share in debates vs newscasts")
plt.xlabel("Theme")
plt.ylabel("Share of known-theme speech time (%)")
plt.xticks(x, plot_df["dominant_theme"], rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "q7_debates_vs_newscasts_theme_share.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Plot: debate share vs newscast share

Each point is one theme.

Themes near the diagonal are similarly represented in debates and newscasts. Themes far from the diagonal are more characteristic of one source.

In [ ]:
scatter_df = comparison_pivot.copy()

plt.figure(figsize=(7, 6))
plt.scatter(scatter_df["Debates"], scatter_df["Newscasts"])

max_val = max(scatter_df["Debates"].max(), scatter_df["Newscasts"].max(), 1)
plt.plot([0, max_val], [0, max_val], linestyle="--")

for _, row in scatter_df.iterrows():
    if row["Debates"] + row["Newscasts"] >= 3:
        plt.text(row["Debates"], row["Newscasts"], str(row["dominant_theme"]), fontsize=8)

plt.title("Q7 — Theme alignment: debates vs newscasts")
plt.xlabel("Debates share (%)")
plt.ylabel("Newscasts share (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "q7_debate_vs_newscast_theme_scatter.png", dpi=160, bbox_inches="tight")
plt.show()

## 13. Debates vs RTP vs TVI

This separates newscasts by channel.

It helps show whether debate topics are more aligned with one channel or another.

In [ ]:
# ============================================================
# Q7 — Theme share by group: Debates vs Newscast channels
# Versão mais legível do heatmap
# ============================================================

channel_windows = comparison_windows.copy()

channel_windows["comparison_group"] = np.where(
    channel_windows["source_family"] == "Debates",
    "Debates",
    "Newscasts " + channel_windows["channel"].fillna("Unknown").astype(str)
)

channel_theme_share = (
    channel_windows
    .groupby(["comparison_group", "dominant_theme"])
    .agg(
        n_windows=("global_window_uid", "count"),
        duration_min=("duration_min", "sum")
    )
    .reset_index()
)

group_totals = (
    channel_theme_share
    .groupby("comparison_group")["duration_min"]
    .sum()
    .rename("group_total_min")
    .reset_index()
)

channel_theme_share = channel_theme_share.merge(
    group_totals,
    on="comparison_group",
    how="left"
)

channel_theme_share["share_pct"] = (
    100
    * channel_theme_share["duration_min"]
    / channel_theme_share["group_total_min"].replace(0, np.nan)
).round(2)

channel_pivot = (
    channel_theme_share
    .pivot_table(
        index="comparison_group",
        columns="dominant_theme",
        values="share_pct",
        fill_value=0
    )
)

preferred_order = [
    "Debates",
    "Newscasts RTP",
    "Newscasts TVI",
    "Newscasts SIC",
    "Newscasts CNN",
    "Newscasts Unknown"
]

ordered_index = (
    [idx for idx in preferred_order if idx in channel_pivot.index]
    + [idx for idx in channel_pivot.index if idx not in preferred_order]
)

channel_pivot = channel_pivot.loc[ordered_index]

# Ordenar temas por peso total
channel_pivot = channel_pivot[
    channel_pivot.sum(axis=0).sort_values(ascending=False).index
]

# Guardar tabelas completas
channel_theme_share.to_csv(
    OUTPUT_DIR / "q7_theme_share_by_group_long.csv",
    index=False
)

channel_pivot.to_csv(
    OUTPUT_DIR / "q7_debates_vs_channels_theme_share_pivot.csv"
)

print("Tabela completa:")
display(channel_pivot)

# ============================================================
# Gráfico legível
# ============================================================

TOP_N_THEMES = 12
# Se quiseres todos os temas, mete:
# TOP_N_THEMES = None

if TOP_N_THEMES is None:
    plot_pivot = channel_pivot.copy()
else:
    top_theme_cols = (
        channel_pivot
        .sum(axis=0)
        .sort_values(ascending=False)
        .head(TOP_N_THEMES)
        .index
    )
    plot_pivot = channel_pivot[top_theme_cols].copy()

# Transpor para os temas ficarem nas linhas
plot_pivot_t = plot_pivot.T

plt.figure(
    figsize=(
        max(8, plot_pivot_t.shape[1] * 1.6),
        max(7, plot_pivot_t.shape[0] * 0.55)
    )
)

plt.imshow(plot_pivot_t.values, aspect="auto")

title_suffix = (
    f"Top {TOP_N_THEMES} themes"
    if TOP_N_THEMES is not None
    else "All themes"
)

plt.title(f"Q7 — Theme share heatmap: Debates vs newscast channels — {title_suffix}")
plt.xlabel("Source group")
plt.ylabel("Theme")

plt.xticks(
    np.arange(plot_pivot_t.shape[1]),
    plot_pivot_t.columns,
    rotation=30,
    ha="right"
)

plt.yticks(
    np.arange(plot_pivot_t.shape[0]),
    plot_pivot_t.index
)

plt.colorbar(label="Share of known-theme speech time (%)")

# Escrever valores dentro das células
for i in range(plot_pivot_t.shape[0]):
    for j in range(plot_pivot_t.shape[1]):
        value = plot_pivot_t.values[i, j]
        if value > 0:
            plt.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "q7_debates_vs_channels_theme_heatmap_readable.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

## 14. Theme profiles per debate and per newscast

This produces per-video summaries that can be used for qualitative interpretation.

In [ ]:
# ------------------------------------------------------------
# Theme profile by video
# Final version:
#   Plot 1 -> Debates
#   Plot 2 -> Newscasts split into RTP and TVI
# Temporal order on Y axis: oldest at top, newest at bottom
# Same colors and same theme order everywhere
# ------------------------------------------------------------

video_theme_share = (
    comparison_windows
    .groupby([
        "video_id", "video_label", "source_family", "source_type",
        "channel", "date", "date_key", "dominant_theme"
    ], dropna=False)
    .agg(
        n_windows=("global_window_uid", "count"),
        duration_min=("duration_min", "sum")
    )
    .reset_index()
)

video_totals = (
    video_theme_share
    .groupby("video_id")["duration_min"]
    .sum()
    .rename("video_known_theme_total_min")
    .reset_index()
)

video_theme_share = video_theme_share.merge(video_totals, on="video_id", how="left")

video_theme_share["share_pct"] = (
    100 * video_theme_share["duration_min"] /
    video_theme_share["video_known_theme_total_min"].replace(0, np.nan)
).round(2)

video_theme_share.to_csv(
    OUTPUT_DIR / "q7_theme_share_by_video_long.csv",
    index=False
)

# ------------------------------------------------------------
# Split groups
# ------------------------------------------------------------

debate_video_share = video_theme_share[
    video_theme_share["source_type"] == "debate"
].copy()

rtp_video_share = video_theme_share[
    (video_theme_share["source_type"] == "newscast") &
    (video_theme_share["channel"].astype(str).str.upper() == "RTP")
].copy()

tvi_video_share = video_theme_share[
    (video_theme_share["source_type"] == "newscast") &
    (video_theme_share["channel"].astype(str).str.upper() == "TVI")
].copy()

# ------------------------------------------------------------
# Prepare dates and labels
# ------------------------------------------------------------

if not debate_video_share.empty:
    debate_video_share["date_sort"] = pd.to_datetime(debate_video_share["date"], errors="coerce")
    debate_video_share["plot_label"] = (
        debate_video_share["video_label"].astype(str)
        + " — "
        + debate_video_share["date_key"].fillna("").astype(str)
    )

if not rtp_video_share.empty:
    rtp_video_share["date_sort"] = pd.to_datetime(rtp_video_share["date"], errors="coerce")
    rtp_video_share["plot_label"] = "RTP — " + rtp_video_share["date_key"].fillna("").astype(str)

if not tvi_video_share.empty:
    tvi_video_share["date_sort"] = pd.to_datetime(tvi_video_share["date"], errors="coerce")
    tvi_video_share["plot_label"] = "TVI — " + tvi_video_share["date_key"].fillna("").astype(str)

# ------------------------------------------------------------
# Pivot tables
# ------------------------------------------------------------

if not debate_video_share.empty:
    debate_pivot = debate_video_share.pivot_table(
        index="plot_label",
        columns="dominant_theme",
        values="share_pct",
        fill_value=0
    )
    debate_order = (
        debate_video_share[["plot_label", "date_sort"]]
        .drop_duplicates()
        .sort_values("date_sort", ascending=True)["plot_label"]
        .tolist()
    )
    debate_pivot = debate_pivot.reindex(debate_order)
else:
    debate_pivot = pd.DataFrame()

if not rtp_video_share.empty:
    rtp_pivot = rtp_video_share.pivot_table(
        index="plot_label",
        columns="dominant_theme",
        values="share_pct",
        fill_value=0
    )
    rtp_order = (
        rtp_video_share[["plot_label", "date_sort"]]
        .drop_duplicates()
        .sort_values("date_sort", ascending=True)["plot_label"]
        .tolist()
    )
    rtp_pivot = rtp_pivot.reindex(rtp_order)
else:
    rtp_pivot = pd.DataFrame()

if not tvi_video_share.empty:
    tvi_pivot = tvi_video_share.pivot_table(
        index="plot_label",
        columns="dominant_theme",
        values="share_pct",
        fill_value=0
    )
    tvi_order = (
        tvi_video_share[["plot_label", "date_sort"]]
        .drop_duplicates()
        .sort_values("date_sort", ascending=True)["plot_label"]
        .tolist()
    )
    tvi_pivot = tvi_pivot.reindex(tvi_order)
else:
    tvi_pivot = pd.DataFrame()

# ------------------------------------------------------------
# Common theme order across all groups
# ------------------------------------------------------------

theme_means = []
for pivot_df in [debate_pivot, rtp_pivot, tvi_pivot]:
    if not pivot_df.empty:
        theme_means.append(pivot_df.mean(axis=0))

combined_theme_mean = pd.concat(theme_means, axis=1).fillna(0).mean(axis=1)
theme_order = combined_theme_mean.sort_values(ascending=False).index.tolist()

if not debate_pivot.empty:
    debate_pivot = debate_pivot.reindex(columns=theme_order, fill_value=0)
    debate_pivot.to_csv(OUTPUT_DIR / "q7_debate_theme_profiles_pivot.csv")

if not rtp_pivot.empty:
    rtp_pivot = rtp_pivot.reindex(columns=theme_order, fill_value=0)
    rtp_pivot.to_csv(OUTPUT_DIR / "q7_rtp_theme_profiles_pivot.csv")

if not tvi_pivot.empty:
    tvi_pivot = tvi_pivot.reindex(columns=theme_order, fill_value=0)
    tvi_pivot.to_csv(OUTPUT_DIR / "q7_tvi_theme_profiles_pivot.csv")

# ------------------------------------------------------------
# Consistent colors
# ------------------------------------------------------------

color_pool = []
for cmap_name in ["tab20", "tab20b", "tab20c"]:
    color_pool.extend(list(plt.get_cmap(cmap_name).colors))

THEME_COLOR_MAP = {
    theme: color_pool[i % len(color_pool)]
    for i, theme in enumerate(theme_order)
}

plot_colors = [THEME_COLOR_MAP[theme] for theme in theme_order]

# ------------------------------------------------------------
# Helper function — plot one theme profile
# ------------------------------------------------------------

def plot_theme_profile(
    pivot_df,
    title,
    ylabel,
    output_png,
    output_svg,
    row_height=0.75,
    fig_width=18
):
    if pivot_df.empty:
        print(f"No data for: {title}")
        return

    fig_height = max(7, len(pivot_df) * row_height)

    ax = pivot_df.plot(
        kind="barh",
        stacked=True,
        figsize=(fig_width, fig_height),
        color=plot_colors,
        width=0.75
    )

    ax.set_title(title, fontsize=15)
    ax.set_xlabel("Share of known-theme time (%)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.tick_params(axis="both", labelsize=10)

    plt.legend(
        title="Theme",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=10,
        title_fontsize=11
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / output_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.savefig(
        OUTPUT_DIR / output_svg,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
plot_theme_profile(
    pivot_df=debate_pivot,
    title="Q7 — Theme profile by debate",
    ylabel="Debate",
    output_png="q7_theme_profile_by_debate_separate.png",
    output_svg="q7_theme_profile_by_debate_separate.svg",
    row_height=0.38,
    fig_width=18
)

In [ ]:
# ------------------------------------------------------------
# Plot combinado — RTP + TVI no mesmo gráfico
# Datas agrupadas em pares RTP/TVI
# Força TODOS os temas na legenda e no plot
# ------------------------------------------------------------

from matplotlib.patches import Patch

if (not rtp_pivot.empty) or (not tvi_pivot.empty):

    # ------------------------------------------------------------
    # Recriar ordem dos temas a partir de TODOS os temas disponíveis
    # Assim não dependemos de theme_order antigo
    # ------------------------------------------------------------

    theme_order_all = (
        video_theme_share
        .dropna(subset=["dominant_theme"])
        .groupby("dominant_theme")["duration_min"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    print("Number of themes in plot:", len(theme_order_all))
    print(theme_order_all)

    # ------------------------------------------------------------
    # Recriar cores para TODOS os temas
    # ------------------------------------------------------------

    color_pool = []
    for cmap_name in ["tab20", "tab20b", "tab20c"]:
        color_pool.extend(list(plt.get_cmap(cmap_name).colors))

    THEME_COLOR_MAP = {
        theme: color_pool[i % len(color_pool)]
        for i, theme in enumerate(theme_order_all)
    }

    plot_colors_all = [
        THEME_COLOR_MAP[theme]
        for theme in theme_order_all
    ]

    # ------------------------------------------------------------
    # Garantir que RTP e TVI têm TODAS as colunas de temas
    # ------------------------------------------------------------

    if not rtp_pivot.empty:
        rtp_pivot_plot = rtp_pivot.reindex(
            columns=theme_order_all,
            fill_value=0
        ).astype(float)
    else:
        rtp_pivot_plot = pd.DataFrame(columns=theme_order_all)

    if not tvi_pivot.empty:
        tvi_pivot_plot = tvi_pivot.reindex(
            columns=theme_order_all,
            fill_value=0
        ).astype(float)
    else:
        tvi_pivot_plot = pd.DataFrame(columns=theme_order_all)

    # ------------------------------------------------------------
    # Criar mapas por data
    # Não perde vídeos se houver mais do que um por data
    # ------------------------------------------------------------

    rtp_by_date = {}
    if not rtp_video_share.empty and not rtp_pivot_plot.empty:
        rtp_meta = (
            rtp_video_share[["plot_label", "date_sort"]]
            .drop_duplicates()
            .sort_values("date_sort")
        )

        for _, row in rtp_meta.iterrows():
            if row["plot_label"] in rtp_pivot_plot.index:
                rtp_by_date.setdefault(row["date_sort"], []).append(row["plot_label"])

    tvi_by_date = {}
    if not tvi_video_share.empty and not tvi_pivot_plot.empty:
        tvi_meta = (
            tvi_video_share[["plot_label", "date_sort"]]
            .drop_duplicates()
            .sort_values("date_sort")
        )

        for _, row in tvi_meta.iterrows():
            if row["plot_label"] in tvi_pivot_plot.index:
                tvi_by_date.setdefault(row["date_sort"], []).append(row["plot_label"])

    # União das datas
    all_dates = sorted(set(rtp_by_date.keys()).union(set(tvi_by_date.keys())))

    combined_rows = []
    combined_index = []

    for d in all_dates:
        date_label = pd.to_datetime(d).strftime("%b_%d")

        # RTP primeiro
        for label in rtp_by_date.get(d, []):
            combined_rows.append(rtp_pivot_plot.loc[label])
            combined_index.append(f"{date_label} — RTP")

        # TVI depois
        for label in tvi_by_date.get(d, []):
            combined_rows.append(tvi_pivot_plot.loc[label])
            combined_index.append(f"{date_label} — TVI")

        # linha vazia para separar datas
        combined_rows.append(pd.Series(0, index=theme_order_all))
        combined_index.append(" ")

    # Criar dataframe final
    combined_pivot = pd.DataFrame(
        combined_rows,
        index=combined_index
    )

    # Remover último espaço vazio
    if len(combined_pivot) > 0 and str(combined_pivot.index[-1]).strip() == "":
        combined_pivot = combined_pivot.iloc[:-1].copy()

    combined_pivot = combined_pivot.reindex(
        columns=theme_order_all,
        fill_value=0
    ).astype(float)

    # Guardar tabela
    combined_pivot.to_csv(
        OUTPUT_DIR / "q7_theme_profile_rtp_tvi_grouped_by_date.csv"
    )

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------

    fig_height = max(8, len(combined_pivot) * 0.5)

    ax = combined_pivot.plot(
        kind="barh",
        stacked=True,
        figsize=(22, fig_height),
        color=plot_colors_all,
        width=0.8,
        legend=False
    )

    ax.set_title("Q7 — Theme profiles over time: RTP vs TVI newscasts")
    ax.set_xlabel("Share of known-theme speech time (%)")
    ax.set_ylabel("Date and channel")
    ax.set_xlim(0, 100)
    ax.invert_yaxis()

    ax.tick_params(axis="y", labelsize=10)
    ax.tick_params(axis="x", labelsize=10)

    # Legenda manual para garantir que TODOS os temas aparecem
    legend_handles = [
        Patch(facecolor=THEME_COLOR_MAP[theme], label=theme)
        for theme in theme_order_all
    ]

    ax.legend(
        handles=legend_handles,
        title="Theme",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10
    )

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    plt.savefig(
        OUTPUT_DIR / "q7_theme_profile_rtp_tvi_grouped_by_date.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.savefig(
        OUTPUT_DIR / "q7_theme_profile_rtp_tvi_grouped_by_date.svg",
        bbox_inches="tight"
    )

    plt.show()

else:
    print("No RTP or TVI data available for combined plot.")

In [ ]:
# ------------------------------------------------------------
# Plot combinado — TODOS os Debates + TODOS os RTP + TODOS os TVI
# Agrupado por data:
#   se houver debate/RTP/TVI na mesma data ficam juntos
#   se houver só telejornal, também aparece
# ------------------------------------------------------------

from matplotlib.patches import Patch

if (not debate_pivot.empty) or (not rtp_pivot.empty) or (not tvi_pivot.empty):

    # ------------------------------------------------------------
    # Garantir date_sort e plot_label
    # ------------------------------------------------------------

    if not debate_video_share.empty:
        debate_video_share["date_sort"] = pd.to_datetime(
            debate_video_share["date"], errors="coerce"
        ).dt.normalize()

        debate_video_share["plot_label"] = (
            debate_video_share["video_label"].astype(str)
            + " — "
            + debate_video_share["date_key"].fillna("").astype(str)
        )

    if not rtp_video_share.empty:
        rtp_video_share["date_sort"] = pd.to_datetime(
            rtp_video_share["date"], errors="coerce"
        ).dt.normalize()

        rtp_video_share["plot_label"] = (
            "RTP — " + rtp_video_share["date_key"].fillna("").astype(str)
        )

    if not tvi_video_share.empty:
        tvi_video_share["date_sort"] = pd.to_datetime(
            tvi_video_share["date"], errors="coerce"
        ).dt.normalize()

        tvi_video_share["plot_label"] = (
            "TVI — " + tvi_video_share["date_key"].fillna("").astype(str)
        )

    # ------------------------------------------------------------
    # Usar TODOS os temas existentes
    # ------------------------------------------------------------

    theme_order_plot = (
        video_theme_share
        .dropna(subset=["dominant_theme"])
        .groupby("dominant_theme")["duration_min"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    print("Number of themes in combined plot:", len(theme_order_plot))
    print(theme_order_plot)

    # ------------------------------------------------------------
    # Garantir cores consistentes
    # ------------------------------------------------------------

    color_pool = []
    for cmap_name in ["tab20", "tab20b", "tab20c"]:
        color_pool.extend(list(plt.get_cmap(cmap_name).colors))

    if "THEME_COLOR_MAP" not in globals():
        THEME_COLOR_MAP = {}

    for theme in theme_order_plot:
        if theme not in THEME_COLOR_MAP:
            THEME_COLOR_MAP[theme] = color_pool[len(THEME_COLOR_MAP) % len(color_pool)]

    plot_colors_plot = [
        THEME_COLOR_MAP[theme]
        for theme in theme_order_plot
    ]

    # ------------------------------------------------------------
    # Garantir mesma ordem de temas em todos os pivots
    # ------------------------------------------------------------

    if not debate_pivot.empty:
        debate_pivot_plot = debate_pivot.reindex(
            columns=theme_order_plot,
            fill_value=0
        ).astype(float)
    else:
        debate_pivot_plot = pd.DataFrame(columns=theme_order_plot)

    if not rtp_pivot.empty:
        rtp_pivot_plot = rtp_pivot.reindex(
            columns=theme_order_plot,
            fill_value=0
        ).astype(float)
    else:
        rtp_pivot_plot = pd.DataFrame(columns=theme_order_plot)

    if not tvi_pivot.empty:
        tvi_pivot_plot = tvi_pivot.reindex(
            columns=theme_order_plot,
            fill_value=0
        ).astype(float)
    else:
        tvi_pivot_plot = pd.DataFrame(columns=theme_order_plot)

    # ------------------------------------------------------------
    # Metadata por data
    # Agora guardamos LISTAS, para não perder vídeos na mesma data
    # ------------------------------------------------------------

    debate_by_date = {}

    if not debate_video_share.empty:
        debate_meta = (
            debate_video_share[
                ["plot_label", "video_label", "date_sort", "date_key"]
            ]
            .drop_duplicates()
            .sort_values(["date_sort", "video_label"])
        )

        for _, row in debate_meta.iterrows():
            d = row["date_sort"]

            if row["plot_label"] in debate_pivot_plot.index:
                debate_by_date.setdefault(d, []).append({
                    "plot_label": row["plot_label"],
                    "video_label": row["video_label"]
                })

    rtp_by_date = {}

    if not rtp_video_share.empty:
        rtp_meta = (
            rtp_video_share[
                ["plot_label", "date_sort", "date_key"]
            ]
            .drop_duplicates()
            .sort_values("date_sort")
        )

        for _, row in rtp_meta.iterrows():
            d = row["date_sort"]

            if row["plot_label"] in rtp_pivot_plot.index:
                rtp_by_date.setdefault(d, []).append(row["plot_label"])

    tvi_by_date = {}

    if not tvi_video_share.empty:
        tvi_meta = (
            tvi_video_share[
                ["plot_label", "date_sort", "date_key"]
            ]
            .drop_duplicates()
            .sort_values("date_sort")
        )

        for _, row in tvi_meta.iterrows():
            d = row["date_sort"]

            if row["plot_label"] in tvi_pivot_plot.index:
                tvi_by_date.setdefault(d, []).append(row["plot_label"])

    # ------------------------------------------------------------
    # Agora usamos TODAS as datas:
    # datas com debates + datas com RTP + datas com TVI
    # ------------------------------------------------------------

    all_dates = sorted(
        set(debate_by_date.keys())
        .union(set(rtp_by_date.keys()))
        .union(set(tvi_by_date.keys()))
    )

    print("Number of dates in combined plot:", len(all_dates))
    print("Debate dates:", len(debate_by_date))
    print("RTP dates:", len(rtp_by_date))
    print("TVI dates:", len(tvi_by_date))

    combined_rows = []
    combined_index = []
    spacer_id = 0

    for d in all_dates:
        date_label = pd.to_datetime(d).strftime("%b_%d")

        # 1) Debates dessa data
        for item in debate_by_date.get(d, []):
            plot_label = item["plot_label"]
            debate_name = str(item["video_label"]).replace("_speech", "")

            row = debate_pivot_plot.loc[plot_label].copy()
            combined_rows.append(row)
            combined_index.append(f"{date_label} — Debate: {debate_name}")

        # 2) RTP dessa data
        for plot_label in rtp_by_date.get(d, []):
            row = rtp_pivot_plot.loc[plot_label].copy()
            combined_rows.append(row)
            combined_index.append(f"{date_label} — RTP")

        # 3) TVI dessa data
        for plot_label in tvi_by_date.get(d, []):
            row = tvi_pivot_plot.loc[plot_label].copy()
            combined_rows.append(row)
            combined_index.append(f"{date_label} — TVI")

        # 4) linha vazia para separar datas
        spacer_id += 1
        combined_rows.append(pd.Series(0, index=theme_order_plot))
        combined_index.append(" " * spacer_id)

    combined_pivot_debate_news = pd.DataFrame(
        combined_rows,
        index=combined_index
    )

    # Remover último espaço vazio
    if len(combined_pivot_debate_news) > 0:
        combined_pivot_debate_news = combined_pivot_debate_news.iloc[:-1].copy()

    combined_pivot_debate_news = combined_pivot_debate_news.reindex(
        columns=theme_order_plot,
        fill_value=0
    ).astype(float)

    # ------------------------------------------------------------
    # Guardar tabela
    # ------------------------------------------------------------

    combined_pivot_debate_news.to_csv(
        OUTPUT_DIR / "q7_theme_profile_all_debates_rtp_tvi_grouped_by_date.csv"
    )

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------

    fig_height = max(10, len(combined_pivot_debate_news) * 0.48)

    ax = combined_pivot_debate_news.plot(
        kind="barh",
        stacked=True,
        figsize=(24, fig_height),
        color=plot_colors_plot,
        width=0.8,
        legend=False
    )

    ax.set_title(
        "Q7 — Theme profiles: all debates and all RTP/TVI newscasts grouped by date",
        fontsize=15
    )

    ax.set_xlabel("Share of known-theme speech time (%)")
    ax.set_ylabel("Date and source")
    ax.set_xlim(0, 100)
    ax.invert_yaxis()

    ax.tick_params(axis="y", labelsize=9)
    ax.tick_params(axis="x", labelsize=10)

    # Legenda manual para garantir que aparecem todos os temas
    legend_handles = [
        Patch(facecolor=THEME_COLOR_MAP[theme], label=theme)
        for theme in theme_order_plot
    ]

    ax.legend(
        handles=legend_handles,
        title="Theme",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10
    )

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    plt.savefig(
        OUTPUT_DIR / "q7_theme_profile_all_debates_rtp_tvi_grouped_by_date.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.savefig(
        OUTPUT_DIR / "q7_theme_profile_all_debates_rtp_tvi_grouped_by_date.svg",
        bbox_inches="tight"
    )

    plt.show()

else:
    print("No debate, RTP, or TVI data available for combined plot.")

In [ ]:
# ------------------------------------------------------------
# Q7 — Debate vs newscast similarity analysis
# Goal:
#   For each debate, compare its topic distribution with every RTP/TVI newscast.
#   Then check whether same-day / previous-day / next-day newscasts
#   are more similar than other newscasts.
# ------------------------------------------------------------

# ------------------------------------------------------------
# Build per-video theme profiles
# ------------------------------------------------------------

profile_pivot = video_theme_share.pivot_table(
    index=[
        "video_id", "video_label", "source_type",
        "channel", "date", "date_key"
    ],
    columns="dominant_theme",
    values="share_pct",
    fill_value=0
).reset_index()

profile_pivot["date"] = pd.to_datetime(
    profile_pivot["date"],
    errors="coerce"
).dt.normalize()

# Normalize channel names
profile_pivot["channel_norm"] = (
    profile_pivot["channel"]
    .fillna("Unknown")
    .astype(str)
    .str.upper()
)

profile_pivot["channel_norm"] = np.where(
    profile_pivot["channel_norm"].str.contains("RTP", na=False),
    "RTP",
    np.where(
        profile_pivot["channel_norm"].str.contains("TVI", na=False),
        "TVI",
        profile_pivot["channel_norm"]
    )
)

# Theme columns
metadata_cols = [
    "video_id", "video_label", "source_type",
    "channel", "channel_norm", "date", "date_key"
]

theme_cols = [
    col for col in profile_pivot.columns
    if col not in metadata_cols
]

# Optional: use same theme order as previous plots if available
if "theme_order_plot" in globals():
    theme_cols = [t for t in theme_order_plot if t in theme_cols]

# Split debates and newscasts
debate_profiles = profile_pivot[
    profile_pivot["source_type"] == "debate"
].copy()

newscast_profiles = profile_pivot[
    profile_pivot["source_type"] == "newscast"
].copy()

# ------------------------------------------------------------
# Similarity functions
# ------------------------------------------------------------

def cosine_similarity_pct(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return np.nan

    return float(np.dot(a, b) / denom)


def topic_distance_pct_points(a, b):
    """
    Total variation distance in percentage points.
    0 = identical topic distribution
    100 = completely different topic distribution
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    return float(0.5 * np.abs(a - b).sum())


def top_k_overlap(a, b, themes, k=5):
    a_series = pd.Series(a, index=themes).sort_values(ascending=False)
    b_series = pd.Series(b, index=themes).sort_values(ascending=False)

    top_a = set(a_series.head(k).index)
    top_b = set(b_series.head(k).index)

    return len(top_a.intersection(top_b))


def top_themes_string(values, themes, k=3):
    s = pd.Series(values, index=themes).sort_values(ascending=False)
    s = s[s > 0].head(k)

    return ", ".join([f"{idx} ({val:.1f}%)" for idx, val in s.items()])


# ------------------------------------------------------------
# Pairwise comparison: every debate vs every newscast
# ------------------------------------------------------------

rows = []

for _, debate in debate_profiles.iterrows():

    debate_vec = debate[theme_cols].values.astype(float)
    debate_date = debate["date"]

    for _, news in newscast_profiles.iterrows():

        news_vec = news[theme_cols].values.astype(float)
        news_date = news["date"]

        day_diff = (news_date - debate_date).days

        if day_diff == -1:
            relation = "previous_day"
        elif day_diff == 0:
            relation = "same_day"
        elif day_diff == 1:
            relation = "next_day"
        else:
            relation = "other_days"

        rows.append({
            "debate_video_id": debate["video_id"],
            "debate_label": debate["video_label"],
            "debate_date": debate_date,
            "debate_date_key": debate["date_key"],

            "newscast_video_id": news["video_id"],
            "newscast_label": news["video_label"],
            "newscast_channel": news["channel_norm"],
            "newscast_date": news_date,
            "newscast_date_key": news["date_key"],

            "day_diff_newscast_minus_debate": day_diff,
            "relation_to_debate": relation,

            "cosine_similarity": cosine_similarity_pct(debate_vec, news_vec),
            "topic_distance_pct_points": topic_distance_pct_points(debate_vec, news_vec),
            "top5_theme_overlap": top_k_overlap(debate_vec, news_vec, theme_cols, k=5),

            "debate_top3_themes": top_themes_string(debate_vec, theme_cols, k=3),
            "newscast_top3_themes": top_themes_string(news_vec, theme_cols, k=3),
        })

similarity_df = pd.DataFrame(rows)

# Rank newscasts for each debate
# rank 1 = most similar newscast to that debate
similarity_df["rank_for_debate"] = (
    similarity_df
    .groupby("debate_video_id")["cosine_similarity"]
    .rank(method="first", ascending=False)
)

similarity_df.to_csv(
    OUTPUT_DIR / "q7_debate_newscast_pairwise_similarity.csv",
    index=False
)

display(Markdown("### Debate vs newscast pairwise similarity"))
display(
    similarity_df
    .sort_values(["debate_date", "rank_for_debate"])
    .head(30)
)

# ------------------------------------------------------------
# Summary: are previous/same/next-day newscasts more similar?
# ------------------------------------------------------------

relation_order = ["previous_day", "same_day", "next_day", "other_days"]

similarity_summary = (
    similarity_df
    .groupby(["relation_to_debate", "newscast_channel"])
    .agg(
        n_pairs=("cosine_similarity", "count"),
        mean_cosine_similarity=("cosine_similarity", "mean"),
        median_cosine_similarity=("cosine_similarity", "median"),
        mean_topic_distance_pct_points=("topic_distance_pct_points", "mean"),
        mean_top5_overlap=("top5_theme_overlap", "mean"),
        mean_rank_for_debate=("rank_for_debate", "mean")
    )
    .reset_index()
)

similarity_summary["relation_to_debate"] = pd.Categorical(
    similarity_summary["relation_to_debate"],
    categories=relation_order,
    ordered=True
)

similarity_summary = similarity_summary.sort_values(
    ["relation_to_debate", "newscast_channel"]
)

similarity_summary.to_csv(
    OUTPUT_DIR / "q7_similarity_summary_previous_same_next_other.csv",
    index=False
)

display(Markdown("### Similarity summary by temporal relation"))
display(similarity_summary)

# ------------------------------------------------------------
# Same-day / previous-day evaluation against other-days baseline
# ------------------------------------------------------------

other_baseline = (
    similarity_df[similarity_df["relation_to_debate"] == "other_days"]
    .groupby(["debate_video_id", "newscast_channel"])
    .agg(
        mean_other_days_cosine=("cosine_similarity", "mean"),
        mean_other_days_distance=("topic_distance_pct_points", "mean")
    )
    .reset_index()
)

nearby_eval = similarity_df[
    similarity_df["relation_to_debate"].isin(["previous_day", "same_day", "next_day"])
].copy()

nearby_eval = nearby_eval.merge(
    other_baseline,
    on=["debate_video_id", "newscast_channel"],
    how="left"
)

nearby_eval["cosine_lift_vs_other_days"] = (
    nearby_eval["cosine_similarity"] -
    nearby_eval["mean_other_days_cosine"]
)

nearby_eval["distance_reduction_vs_other_days"] = (
    nearby_eval["mean_other_days_distance"] -
    nearby_eval["topic_distance_pct_points"]
)

nearby_eval.to_csv(
    OUTPUT_DIR / "q7_nearby_newscast_lift_vs_other_days.csv",
    index=False
)

display(Markdown("### Nearby newscasts compared with other-days baseline"))
display(
    nearby_eval
    .sort_values(["debate_date", "relation_to_debate", "newscast_channel"])
    .head(60)
)

# ------------------------------------------------------------
# Best matches per debate
# ------------------------------------------------------------

best_matches = (
    similarity_df
    .sort_values(["debate_video_id", "cosine_similarity"], ascending=[True, False])
    .groupby("debate_video_id")
    .head(5)
    .copy()
)

best_matches.to_csv(
    OUTPUT_DIR / "q7_best_newscast_matches_per_debate.csv",
    index=False
)

display(Markdown("### Best newscast matches per debate"))
display(
    best_matches[
        [
            "debate_date_key",
            "debate_label",
            "newscast_date_key",
            "newscast_channel",
            "relation_to_debate",
            "cosine_similarity",
            "topic_distance_pct_points",
            "rank_for_debate",
            "debate_top3_themes",
            "newscast_top3_themes"
        ]
    ]
)

# ------------------------------------------------------------
# Plot 1: mean cosine similarity by temporal relation
# ------------------------------------------------------------

summary_plot = similarity_summary.pivot_table(
    index="relation_to_debate",
    columns="newscast_channel",
    values="mean_cosine_similarity",
    fill_value=0
).reindex(relation_order)

ax = summary_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Q7 — Debate/newscast topic similarity by temporal relation")
ax.set_xlabel("Newscast timing relative to debate")
ax.set_ylabel("Mean cosine similarity")
ax.set_ylim(0, 1)

plt.xticks(rotation=0)
plt.legend(title="Channel")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "q7_similarity_by_temporal_relation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Plot 2: distance by temporal relation
# Lower is better
# ------------------------------------------------------------

distance_plot = similarity_summary.pivot_table(
    index="relation_to_debate",
    columns="newscast_channel",
    values="mean_topic_distance_pct_points",
    fill_value=0
).reindex(relation_order)

ax = distance_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Q7 — Debate/newscast topic distance by temporal relation")
ax.set_xlabel("Newscast timing relative to debate")
ax.set_ylabel("Mean topic distance percentage points")

plt.xticks(rotation=0)
plt.legend(title="Channel")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "q7_distance_by_temporal_relation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 15. Temporal relation: before / same day / after debates

This chapter explores whether debate themes are more visible in newscasts before, on the same day, or after the debate date.

The interpretation should be careful:

```text
This suggests temporal alignment, not causality.
```

In [ ]:
temporal_windows = comparison_windows.dropna(subset=["date"]).copy()

debate_dates = sorted(temporal_windows.loc[temporal_windows["source_type"] == "debate", "date"].dropna().unique())
newscast_dates = sorted(temporal_windows.loc[temporal_windows["source_type"] == "newscast", "date"].dropna().unique())

print("Debate dates found:", [pd.Timestamp(d).strftime("%Y-%m-%d") for d in debate_dates])
print("Newscast dates found:", [pd.Timestamp(d).strftime("%Y-%m-%d") for d in newscast_dates])

# Daily theme table.
daily_theme = (
    temporal_windows
    .groupby(["date", "date_key", "source_family", "source_type", "channel", "dominant_theme"])
    .agg(n_windows=("global_window_uid", "count"), duration_min=("duration_min", "sum"))
    .reset_index()
)

daily_theme.to_csv(OUTPUT_DIR / "q7_daily_theme_duration.csv", index=False)

all_themes = sorted(temporal_windows["dominant_theme"].dropna().unique())
temporal_rows = []

for debate_date in debate_dates:
    debate_date = pd.Timestamp(debate_date)
    before_start = debate_date - timedelta(days=TEMPORAL_WINDOW_DAYS)
    before_end = debate_date - timedelta(days=1)
    after_start = debate_date + timedelta(days=1)
    after_end = debate_date + timedelta(days=TEMPORAL_WINDOW_DAYS)

    debate_day = temporal_windows[
        (temporal_windows["source_type"] == "debate") &
        (temporal_windows["date"] == debate_date)
    ]

    newscasts_before = temporal_windows[
        (temporal_windows["source_type"] == "newscast") &
        (temporal_windows["date"] >= before_start) &
        (temporal_windows["date"] <= before_end)
    ]

    newscasts_same = temporal_windows[
        (temporal_windows["source_type"] == "newscast") &
        (temporal_windows["date"] == debate_date)
    ]

    newscasts_after = temporal_windows[
        (temporal_windows["source_type"] == "newscast") &
        (temporal_windows["date"] >= after_start) &
        (temporal_windows["date"] <= after_end)
    ]

    for theme in all_themes:
        debate_min = debate_day.loc[debate_day["dominant_theme"] == theme, "duration_min"].sum()
        before_min = newscasts_before.loc[newscasts_before["dominant_theme"] == theme, "duration_min"].sum()
        same_min = newscasts_same.loc[newscasts_same["dominant_theme"] == theme, "duration_min"].sum()
        after_min = newscasts_after.loc[newscasts_after["dominant_theme"] == theme, "duration_min"].sum()

        temporal_rows.append({
            "debate_date": debate_date,
            "debate_date_key": date_key_from_timestamp(debate_date),
            "theme": theme,
            "debate_duration_min_on_date": debate_min,
            f"newscast_duration_min_before_{TEMPORAL_WINDOW_DAYS}d": before_min,
            "newscast_duration_min_same_day": same_min,
            f"newscast_duration_min_after_{TEMPORAL_WINDOW_DAYS}d": after_min,
            "after_minus_before_min": after_min - before_min,
        })

temporal_relation = pd.DataFrame(temporal_rows)

if not temporal_relation.empty:
    temporal_relation = temporal_relation[
        (temporal_relation["debate_duration_min_on_date"] > 0) |
        (temporal_relation[f"newscast_duration_min_before_{TEMPORAL_WINDOW_DAYS}d"] > 0) |
        (temporal_relation["newscast_duration_min_same_day"] > 0) |
        (temporal_relation[f"newscast_duration_min_after_{TEMPORAL_WINDOW_DAYS}d"] > 0)
    ].copy()

    temporal_relation.to_csv(OUTPUT_DIR / "q7_temporal_before_same_after_by_debate_date_theme.csv", index=False)

    display(Markdown("### Temporal before/same/after table"))
    display(temporal_relation.sort_values(["debate_date", "debate_duration_min_on_date"], ascending=[True, False]).head(60))

    temporal_summary_by_theme = (
        temporal_relation
        .groupby("theme")
        .agg(
            debate_duration_min_on_debate_dates=("debate_duration_min_on_date", "sum"),
            newscast_before_min=(f"newscast_duration_min_before_{TEMPORAL_WINDOW_DAYS}d", "sum"),
            newscast_same_day_min=("newscast_duration_min_same_day", "sum"),
            newscast_after_min=(f"newscast_duration_min_after_{TEMPORAL_WINDOW_DAYS}d", "sum"),
            after_minus_before_min=("after_minus_before_min", "sum"),
        )
        .reset_index()
        .sort_values("debate_duration_min_on_debate_dates", ascending=False)
    )

    temporal_summary_by_theme.to_csv(OUTPUT_DIR / "q7_temporal_summary_by_theme.csv", index=False)

    display(Markdown("### Temporal summary by theme"))
    display(temporal_summary_by_theme)

else:
    temporal_summary_by_theme = pd.DataFrame()
    print("Temporal relation table is empty. This usually means dates could not be parsed or no debate/newscast dates overlap nearby.")

## 16. Temporal plot

This plot aggregates across debate dates and compares newscast coverage before, same day and after for the main debate themes.

In [ ]:
# ------------------------------------------------------------
# Q7 case study:
# Select one debate and compare its themes with RTP/TVI newscasts
# from the same day and previous day
# ------------------------------------------------------------

# Choose one of these options:
SELECTED_DEBATE_VIDEO_ID = None
SELECTED_DEBATE_LABEL_CONTAINS = ""   # example: "Costa" or part of debate name
SELECTED_DEBATE_DATE = "2025-12-02"           # example: "2024-12-02"

TOP_N_THEMES = 12
INCLUDE_PREVIOUS_DAY = True
INCLUDE_SAME_DAY = True
INCLUDE_ALL_NEWSCAST_BASELINE = True

cw = comparison_windows.dropna(subset=["date"]).copy()
cw["date"] = pd.to_datetime(cw["date"], errors="coerce").dt.normalize()

# Normalize channel names
cw["channel_norm"] = (
    cw["channel"]
    .fillna("Unknown")
    .astype(str)
    .str.upper()
)

cw["channel_norm"] = np.where(
    cw["channel_norm"].str.contains("RTP", na=False),
    "RTP",
    np.where(
        cw["channel_norm"].str.contains("TVI", na=False),
        "TVI",
        cw["channel_norm"]
    )
)

# ------------------------------------------------------------
# Show available debates
# ------------------------------------------------------------

debates_meta = (
    cw[cw["source_type"] == "debate"]
    [["video_id", "video_label", "date", "date_key"]]
    .drop_duplicates()
    .sort_values("date")
)

display(Markdown("### Available debates"))
display(debates_meta)

# ------------------------------------------------------------
# Select debate
# ------------------------------------------------------------

selected_debates = debates_meta.copy()

if SELECTED_DEBATE_VIDEO_ID is not None:
    selected_debates = selected_debates[
        selected_debates["video_id"] == SELECTED_DEBATE_VIDEO_ID
    ]

elif SELECTED_DEBATE_LABEL_CONTAINS.strip() != "":
    selected_debates = selected_debates[
        selected_debates["video_label"]
        .astype(str)
        .str.contains(SELECTED_DEBATE_LABEL_CONTAINS, case=False, na=False)
    ]

elif SELECTED_DEBATE_DATE is not None:
    selected_date = pd.to_datetime(SELECTED_DEBATE_DATE).normalize()
    selected_debates = selected_debates[
        selected_debates["date"] == selected_date
    ]

else:
    # default: first debate in temporal order
    selected_debates = selected_debates.head(1)

if selected_debates.empty:
    raise ValueError("No debate matched your selection. Check video_id, label, or date.")

selected_debate = selected_debates.iloc[0]

selected_video_id = selected_debate["video_id"]
selected_label = selected_debate["video_label"]
selected_date = pd.to_datetime(selected_debate["date"]).normalize()
previous_date = selected_date - pd.Timedelta(days=1)

print("Selected debate:")
print("Video ID:", selected_video_id)
print("Label:", selected_label)
print("Date:", selected_date.strftime("%Y-%m-%d"))
print("Previous day:", previous_date.strftime("%Y-%m-%d"))

# ------------------------------------------------------------
# Helper: theme profile
# ------------------------------------------------------------

def theme_profile(df, group_label):
    out = (
        df.groupby("dominant_theme")
        .agg(
            duration_min=("duration_min", "sum"),
            n_windows=("global_window_uid", "count")
        )
        .reset_index()
    )

    total = out["duration_min"].sum()

    if total > 0:
        out["share_pct"] = (100 * out["duration_min"] / total).round(2)
    else:
        out["share_pct"] = 0

    out["group"] = group_label
    out["total_group_duration_min"] = round(total, 2)

    return out

# ------------------------------------------------------------
# Build comparison groups
# ------------------------------------------------------------

profiles = []

# Selected debate
debate_df = cw[
    (cw["source_type"] == "debate") &
    (cw["video_id"] == selected_video_id)
]

profiles.append(
    theme_profile(debate_df, "Selected debate")
)

# Previous day newscasts
if INCLUDE_PREVIOUS_DAY:
    for channel in ["RTP", "TVI"]:
        df = cw[
            (cw["source_type"] == "newscast") &
            (cw["channel_norm"] == channel) &
            (cw["date"] == previous_date)
        ]

        if not df.empty:
            profiles.append(
                theme_profile(df, f"{channel} previous day")
            )

# Same day newscasts
if INCLUDE_SAME_DAY:
    for channel in ["RTP", "TVI"]:
        df = cw[
            (cw["source_type"] == "newscast") &
            (cw["channel_norm"] == channel) &
            (cw["date"] == selected_date)
        ]

        if not df.empty:
            profiles.append(
                theme_profile(df, f"{channel} same day")
            )

# Baseline: all newscasts
if INCLUDE_ALL_NEWSCAST_BASELINE:
    all_news_df = cw[
        cw["source_type"] == "newscast"
    ]

    profiles.append(
        theme_profile(all_news_df, "All newscasts baseline")
    )

case_profiles = pd.concat(profiles, ignore_index=True)

# ------------------------------------------------------------
# Pivot table
# ------------------------------------------------------------

case_pivot = case_profiles.pivot_table(
    index="dominant_theme",
    columns="group",
    values="share_pct",
    fill_value=0
)

# Order themes by importance in selected debate
theme_order = (
    case_pivot["Selected debate"]
    .sort_values(ascending=False)
    .head(TOP_N_THEMES)
    .index
)

case_pivot_top = case_pivot.loc[theme_order]

case_pivot_top.to_csv(
    OUTPUT_DIR / "q7_case_study_selected_debate_vs_newscasts.csv"
)

display(Markdown("### Selected debate vs nearby newscasts"))
display(case_pivot_top)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

ax = case_pivot_top.plot(
    kind="bar",
    figsize=(14, 6)
)

ax.set_title(
    "Q7 — Selected debate vs RTP/TVI newscasts around the debate date"
)
ax.set_xlabel("Theme")
ax.set_ylabel("Share of known-theme speech time (%)")

plt.xticks(rotation=45, ha="right")
plt.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "q7_case_study_selected_debate_vs_newscasts.png",
    dpi=160,
    bbox_inches="tight"
)

plt.show()

## 17. Draft summary for the report/video

This cell writes a draft textual answer for Question 7.

In [ ]:
n_debate_videos = int((video_summary["source_type"] == "debate").sum()) if "source_type" in video_summary else 0
n_newscast_videos = int((video_summary["source_type"] == "newscast").sum()) if "source_type" in video_summary else 0
n_total_windows = len(all_windows)
n_comparison_windows = len(comparison_windows)

metric_line = ""
if not metrics.empty:
    metric_row = metrics.iloc[0]
    metric_line = (
        f"The theme overlap between debates and newscasts was measured with "
        f"Jaccard={metric_row['jaccard_overlap']:.2f} and cosine similarity="
        f"{metric_row['cosine_similarity_theme_shares']:.2f}."
    )

common_theme_lines = []
for _, row in comparison_pivot[comparison_pivot["relation"] == "present_in_both"].head(8).iterrows():
    common_theme_lines.append(
        f"- {row['dominant_theme']}: Debates {row['Debates']:.1f}%, Newscasts {row['Newscasts']:.1f}%"
    )

more_debate_lines = []
for _, row in comparison_pivot.sort_values("Debates", ascending=False).head(5).iterrows():
    more_debate_lines.append(f"- {row['dominant_theme']}: {row['Debates']:.1f}% in debates")

more_newscast_lines = []
for _, row in comparison_pivot.sort_values("Newscasts", ascending=False).head(5).iterrows():
    more_newscast_lines.append(f"- {row['dominant_theme']}: {row['Newscasts']:.1f}% in newscasts")

temporal_lines = []
if 'temporal_summary_by_theme' in globals() and not temporal_summary_by_theme.empty:
    tmp = temporal_summary_by_theme.sort_values("debate_duration_min_on_debate_dates", ascending=False).head(5)
    for _, row in tmp.iterrows():
        temporal_lines.append(
            f"- {row['theme']}: before={row['newscast_before_min']:.1f} min, "
            f"same day={row['newscast_same_day_min']:.1f} min, "
            f"after={row['newscast_after_min']:.1f} min"
        )
else:
    temporal_lines.append("- Temporal analysis could not be completed because dates were missing or insufficient.")

summary_text = f"""
# Q7 Draft Summary — Debate topics vs Newscast topics

To relate the topics discussed in debates with those identified in newscasts, we used a speech-only approach for both video types. All `*_speech.pkl` files were processed with the same method: transcripts were divided into fixed 30-second windows, each window was assigned a dominant theme using the same topic dictionary, and the percentage of spoken time per theme was compared between debates and newscasts.

Input processed:

- Debate videos: {n_debate_videos}
- Newscast videos: {n_newscast_videos}
- Total 30-second speech windows: {n_total_windows}
- Known-theme windows used in the main comparison: {n_comparison_windows}

{metric_line}

Themes present in both debates and newscasts:

{chr(10).join(common_theme_lines) if common_theme_lines else "No common known themes were found with the current dictionary."}

Most represented themes in debates:

{chr(10).join(more_debate_lines)}

Most represented themes in newscasts:

{chr(10).join(more_newscast_lines)}

Temporal exploration:

For each debate date, we compared newscast coverage in the {TEMPORAL_WINDOW_DAYS} days before, on the same day, and in the {TEMPORAL_WINDOW_DAYS} days after the debate. This analysis is exploratory and indicates temporal alignment, not causality.

{chr(10).join(temporal_lines)}

Interpretation:

The results show whether the debate agenda and the newscast agenda overlap thematically. Themes that appear in both sources suggest shared public or political salience. Themes that are stronger in debates may reflect candidate-driven or programmatic discussion, while themes stronger in newscasts may reflect broader news events. The temporal analysis can suggest whether some themes were already prominent in newscasts before debates or became more visible afterwards, but it should not be interpreted as proof that one caused the other.
""".strip()

summary_path = OUTPUT_DIR / "q7_draft_summary.md"
summary_path.write_text(summary_text, encoding="utf-8")

display(Markdown(summary_text))
print("Saved:", summary_path)

## 18. Output checklist

Main outputs:

```text
q7_all_speech_30s_windows.csv
q7_video_processing_summary.csv
q7_debates_vs_newscasts_theme_share_pivot.csv
q7_debate_newscast_overlap_metrics.csv
q7_debates_vs_channels_theme_share_pivot.csv
q7_temporal_before_same_after_by_debate_date_theme.csv
q7_draft_summary.md
```

Main plots:

```text
q7_debates_vs_newscasts_theme_share.png
q7_debate_vs_newscast_theme_scatter.png
q7_debates_vs_channels_theme_heatmap.png
q7_theme_profile_by_debate.png
q7_temporal_newscast_before_same_after.png
```

In [ ]:
print("Outputs saved in:", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)